# MERFISH 留出切片验证

第二个 cell 记录留出编号。默认依次留出中间的 `1、2、3` 号切片；每轮使用其余 4 张切片，从头训练一个独立模型。

这是基于当前代码实现的留出验证方案，不代表论文原文采用了完全相同的验证协议。沿用本项目教程的切片间距 50 µm，以及此前运行的主要训练参数。

- 留出切片不参与训练、坐标归一化、UOT 配对、标签编码和生成数量设定。训练和生成完成后才读取留出切片真值。
- 只重建留出切片两侧参考切片之间的区间。使用固定宽度 10 µm 的预测薄层与留出真值比较，不按真值细胞数筛选生成细胞。
- 三轮使用相同随机种子、训练轮数和参数；固定使用最后一轮模型的 EMA 权重生成，不用留出指标选择 checkpoint。
- 结果写入 `output/merfish_holdout/运行时间/holdout_编号/`。原有 `output` 结果保留。


In [2]:
# 留出切片编号专用 cell：只需修改这一行。
# 单轮示例：[2]；连续三轮：[1, 2, 3]。编号与 merfish_编号.h5ad 一致，从 0 开始。
HOLDOUT_SLICE_IDS = [1, 2, 3]


## 运行环境

在服务器的 `deepspatial` 环境中打开本文件，选择对应 Python 内核，然后按顺序运行全部 cell。

如果 VS Code 提示缺少内核依赖，先在服务器终端执行：

```bash
conda activate deepspatial
python -m pip install ipykernel
```

### 让 Notebook 内核继承 CUDA 库路径

此前本环境曾出现 `libcusparse.so.12` 找不到 `__nvJitLinkComplete_12_4` 的错误。
终端里的 `export` 不一定传给 VS Code 单独启动的 Notebook 内核；仅在运行中的 cell 修改此变量也不能可靠改变 Linux 动态库搜索路径。
若错误再次出现，应在服务器终端设置路径，并写入内核启动配置：

```bash
conda activate deepspatial
export LD_LIBRARY_PATH="$CONDA_PREFIX/lib/python3.10/site-packages/nvidia/nvjitlink/lib${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"
python -c "import torch; print(torch.__version__); print(torch.cuda.is_available())"
python -m ipykernel install --user --name deepspatial --display-name "Python (deepspatial)" --env LD_LIBRARY_PATH "$LD_LIBRARY_PATH"
```

随后在 VS Code 重新选择上面注册的 **Python (deepspatial)** Jupyter 内核，关闭旧内核并启动新内核，再从头运行。
如内核列表仍显示旧信息，可执行 VS Code 的 **Developer: Reload Window** 后重新选择。
需要选择注册的 Jupyter 内核，才能使用该内核配置中的环境变量。
如果终端测试也失败，请保留完整回溯最底部的 `ImportError: ...` 行，不要仅根据 `torch._C` 的报错位置判断原因。

默认使用物理 GPU 0（RTX 3090）。修改 GPU 编号后需重启内核。


In [3]:
import os
import sys
import json
import gc
import time
from pathlib import Path
from datetime import datetime

# 从项目根目录、教程目录或它们的子目录运行均可。
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "deepspatial" / "core.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请在 DeepSpatial 项目目录下打开此 Notebook。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "merfish_mouse_hypothalamus"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "merfish_holdout"
GPU_INDEX = 0
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_INDEX)  # 在导入 torch 前设置

SEED = 42
EPOCHS = 100
BATCH_SIZE = 512
NUM_WORKERS = 0  # Notebook 默认单进程加载数据
N_SAMPLES_BASE = 50000
SLICE_GAP_UM = 50.0
THICKNESS_UM = 10.0
SLAB_HALF_WIDTH_UM = 5.0
ODE_STEPS = 100
CHUNK_SIZE = 512
LABEL_KEY = "cell_class"
RARE_TRAIN_FRACTION = 0.01  # 只根据训练切片定义稀有类：占比 < 1%
COVERAGE_RADII_UM = (25, 50, 100)

DATA_KWARGS = dict(
    spatial_key="spatial", z_key="z_coord", label_key=LABEL_KEY,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, n_samples_base=N_SAMPLES_BASE,
    alpha_spatial=0.5, uot_reg=0.8, uot_tau=0.05, mode="fit",
)
MODEL_KWARGS = dict(
    patch_size=8, hidden_size=256, depth=6, num_heads=8, mlp_ratio=4.0,
    path_type="Linear", lr=2e-4, weight_decay=1e-5, lambda_g=0.1, lambda_c=10.0,
    sampling_method="dopri5", atol=1e-5, rtol=1e-5,
)
assert HOLDOUT_SLICE_IDS and len(set(HOLDOUT_SLICE_IDS)) == len(HOLDOUT_SLICE_IDS)
assert all(type(i) is int and i in (1, 2, 3) for i in HOLDOUT_SLICE_IDS), "只允许留出中间编号 1、2、3"
assert EPOCHS > 0 and BATCH_SIZE > 0 and CHUNK_SIZE > 0 and ODE_STEPS >= 2
assert 0 < SLAB_HALF_WIDTH_UM < SLICE_GAP_UM and THICKNESS_UM > 0
assert 0 < RARE_TRAIN_FRACTION < 1 and all(r > 0 for r in COVERAGE_RADII_UM)
print("项目：", PROJECT_ROOT)
print("留出顺序：", HOLDOUT_SLICE_IDS)
print("结果根目录：", OUTPUT_ROOT)


项目： /private_sdb/huangyingjie/Code/DeepSpatial
留出顺序： [1, 2, 3]
结果根目录： /private_sdb/huangyingjie/Code/DeepSpatial/output/merfish_holdout


In [4]:
import re
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from scipy.spatial import cKDTree
from IPython.display import display


In [5]:
def find_slice_files(data_dir):
    # 优先直接目录；兼容本地解压后多一层目录的结构，遇到重复编号则报错。
    paths = list(data_dir.glob("merfish_*.h5ad"))
    if not paths:
        paths = list(data_dir.rglob("merfish_*.h5ad"))
    files = {}
    for path in sorted(paths):
        match = re.fullmatch(r"merfish_(\d+)\.h5ad", path.name)
        if match is None:
            continue
        index = int(match.group(1))
        if index in files:
            raise ValueError(f"切片 {index} 出现重复文件，请明确 DATA_DIR：{files[index]} / {path}")
        files[index] = path
    if set(files) != set(range(5)):
        raise FileNotFoundError(f"需要 merfish_0.h5ad 到 merfish_4.h5ad，当前找到：{sorted(files)}；目录：{data_dir}")
    return files


def training_ids(heldout_id):
    if heldout_id not in (1, 2, 3):
        raise ValueError("留出编号必须是 1、2 或 3")
    return [i for i in range(5) if i != heldout_id]


def read_slice(index):
    a = ad.read_h5ad(SLICE_FILES[index])
    if a.n_obs == 0 or not a.var_names.is_unique:
        raise ValueError(f"切片 {index} 为空或基因名重复")
    if LABEL_KEY not in a.obs or a.obs[LABEL_KEY].isna().any():
        raise ValueError(f"切片 {index} 的 {LABEL_KEY} 缺失")
    xy = np.asarray(a.obsm["spatial"], dtype=np.float64)
    values = a.X.data if sp.issparse(a.X) else np.asarray(a.X)
    if xy.shape != (a.n_obs, 2) or not np.isfinite(xy).all():
        raise ValueError(f"切片 {index} 的空间坐标无效")
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f"切片 {index} 的表达值应为有限非负值")
    a.obsm["spatial"] = xy.copy()
    a.obs[LABEL_KEY] = a.obs[LABEL_KEY].astype(str)
    a.obs["input_slice_id"] = index
    a.obs["z_coord"] = float(index * SLICE_GAP_UM)
    return a


def add_spatial_3d(a):
    a.obsm["spatial_3d"] = np.column_stack([a.obsm["spatial"], a.obs["z_coord"].to_numpy()])
    return a


SLICE_FILES = find_slice_files(DATA_DIR)
display(pd.DataFrame([
    {"heldout_id": i, "train_ids": training_ids(i), "reference_pair": [i - 1, i + 1],
     "heldout_z_um": i * SLICE_GAP_UM}
    for i in HOLDOUT_SLICE_IDS
]))


,heldout_id,train_ids,reference_pair,heldout_z_um
0,1,"[0, 2, 3, 4]","[0, 2]",50.0
1,2,"[0, 1, 3, 4]","[1, 3]",100.0
2,3,"[0, 1, 2, 4]","[2, 4]",150.0


In [6]:
# 此 cell 检查实际训练所需的依赖和 GPU；未启动训练。
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from deepspatial import DeepSpatial

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 CUDA GPU，请选择服务器上的 deepspatial 内核并检查环境。")
x = torch.randn(64, 64, device="cuda:0")
_ = x @ x
torch.cuda.synchronize()
del x, _
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("可见 GPU 0:", torch.cuda.get_device_name(0))


PyTorch: 2.5.1+cu124 CUDA: 12.4
可见 GPU 0: NVIDIA GeForce RTX 3090


## 评估定义

生成细胞与真实细胞没有一一对应关系，因此不按行计算细胞分类准确率或表达 MSE。

- **数量与比例**：逐类记录训练、真值和预测数量，以及预测相对真值的比例差。`missing_in_prediction` 表示真值存在、预测为零。
- **稀有类**：本轮训练四张切片中占比低于 1% 的类别。真值中出现而训练中没有的类别另标为 `unseen_in_training`，不会偷偷加入模型标签表。
- **空间覆盖率**：真值某类细胞中，在 XY 平面指定半径内能找到同类预测细胞的比例（`coverage_recall_*um`）；同时报告预测到真值的 `coverage_precision_*um`。这是点集覆盖率，允许多个点共用邻居，不是逐细胞配对召回率。固定报告 25、50、100 µm；空间指标受生成密度和半径影响。
- **表达相关性**：对每个细胞先做 `log1p(max(X, 0))`，再按类别计算平均表达向量的 Pearson 相关性；缺失类或常数向量记为空值。原始生成表达不裁剪，额外报告负值比例。
- **整体几何误差**：不区分类别时，真值到预测、预测到真值两组最近邻距离均值的平均值，单位 µm，越低越好。

稀有类宏平均只纳入真值中存在的训练稀有类；其中完全没生成的类别覆盖召回率按 0 计入；没有预测点时覆盖精确率未定义，保留为空值。训练未见类数量另外报告。
`neuron_class`、`Region` 等字段是源细胞继承信息，预测文件将其改名为 `source_*`，不得当作模型预测标签评价。


In [7]:
def log_expression(a):
    x = a.X.toarray() if sp.issparse(a.X) else np.asarray(a.X)
    if not np.isfinite(x).all():
        raise ValueError("表达矩阵存在 NaN/Inf，停止计算指标")
    return np.log1p(np.maximum(x, 0)).astype(np.float64)


def pearson_or_nan(a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def nearest_distances(query, reference):
    if len(query) == 0:
        return np.empty(0)
    if len(reference) == 0:
        return np.full(len(query), np.inf)
    return cKDTree(reference).query(query, k=1)[0]


def evaluate_holdout(truth, prediction, train_counts):
    if not truth.var_names.equals(prediction.var_names):
        raise ValueError("真值与预测基因顺序不一致")
    if truth.n_obs == 0 or prediction.n_obs == 0:
        raise ValueError("真值或预测薄层为空，不能计算整体指标")
    truth_labels = truth.obs[LABEL_KEY].astype(str).to_numpy()
    pred_labels = prediction.obs[LABEL_KEY].astype(str).to_numpy()
    truth_xy = np.asarray(truth.obsm["spatial"])
    pred_xy = np.asarray(prediction.obsm["spatial"])
    if not np.isfinite(truth_xy).all() or not np.isfinite(pred_xy).all():
        raise ValueError("真值或预测空间坐标存在 NaN/Inf")
    truth_g, pred_g = log_expression(truth), log_expression(prediction)
    classes = sorted(set(train_counts.index) | set(truth_labels) | set(pred_labels))
    rows = []
    for label in classes:
        tmask, pmask = truth_labels == label, pred_labels == label
        nt, npred, ntrain = int(tmask.sum()), int(pmask.sum()), int(train_counts.get(label, 0))
        train_fraction = ntrain / int(train_counts.sum())
        t2p = nearest_distances(truth_xy[tmask], pred_xy[pmask])
        p2t = nearest_distances(pred_xy[pmask], truth_xy[tmask])
        row = dict(
            cell_class=label, train_n=ntrain, train_fraction=train_fraction,
            is_rare_train=bool(ntrain > 0 and train_fraction < RARE_TRAIN_FRACTION),
            truth_n=nt, predicted_n=npred,
            truth_fraction=nt / truth.n_obs, predicted_fraction=npred / prediction.n_obs,
            fraction_error_pp=100 * (npred / prediction.n_obs - nt / truth.n_obs),
            missing_in_prediction=bool(nt > 0 and npred == 0),
            unseen_in_training=bool(nt > 0 and ntrain == 0),
            truth_to_pred_nn_mean_um=float(t2p.mean()) if nt and npred else float("nan"),
            pred_to_truth_nn_mean_um=float(p2t.mean()) if nt and npred else float("nan"),
            mean_log_expression_pcc=pearson_or_nan(truth_g[tmask].mean(0), pred_g[pmask].mean(0))
                                    if nt and npred else float("nan"),
        )
        for radius in COVERAGE_RADII_UM:
            row[f"coverage_recall_{radius}um"] = float(np.mean(t2p <= radius)) if nt else float("nan")
            row[f"coverage_precision_{radius}um"] = float(np.mean(p2t <= radius)) if npred else float("nan")
        rows.append(row)
    table = pd.DataFrame(rows)
    rare = table[table.is_rare_train & (table.truth_n > 0)]
    all_t2p = nearest_distances(truth_xy, pred_xy)
    all_p2t = nearest_distances(pred_xy, truth_xy)
    pred_values = prediction.X.data if sp.issparse(prediction.X) else np.asarray(prediction.X)
    summary = dict(
        truth_n=int(truth.n_obs), predicted_n=int(prediction.n_obs),
        predicted_to_truth_count_ratio=float(prediction.n_obs / truth.n_obs),
        truth_classes=int(len(set(truth_labels))), predicted_classes=int(len(set(pred_labels))),
        missing_classes=int(table.missing_in_prediction.sum()),
        unseen_training_classes=int(table.unseen_in_training.sum()),
        rare_classes_present_in_truth=int(len(rare)),
        rare_missing_classes=int(rare.missing_in_prediction.sum()),
        symmetric_mean_nn_um=float((all_t2p.mean() + all_p2t.mean()) / 2),
        global_mean_log_expression_pcc=pearson_or_nan(truth_g.mean(0), pred_g.mean(0)),
        negative_prediction_fraction=float(np.count_nonzero(pred_values < 0) / (prediction.n_obs * prediction.n_vars)),
    )
    for radius in COVERAGE_RADII_UM:
        summary[f"rare_macro_coverage_recall_{radius}um"] = float(rare[f"coverage_recall_{radius}um"].mean())
        summary[f"rare_macro_coverage_precision_{radius}um"] = float(rare[f"coverage_precision_{radius}um"].mean())
    return table, summary


def write_json(path, data):
    # JSON 中未定义的相关性/宏平均写 null；CSV 中保持为空值。
    clean = {k: None if isinstance(v, (float, np.floating)) and not np.isfinite(v) else v
             for k, v in data.items()}
    path.write_text(json.dumps(clean, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


## 单轮执行函数

为保存逐轮训练损失 CSV，下面直接使用当前 `DeepSpatial.fit()` 内部使用的 Lightning Trainer，并增加 CSVLogger。
模型、损失、UOT 与生成接口仍调用项目实现。保留训练损失最小 checkpoint 和 `last.ckpt`；生成使用训练结束时的最后模型，而非用留出真值挑选模型。

每轮先完成四张训练切片的训练及区间生成，之后再读取第五张真值切片。区间和预测薄层都会保存，便于后续统一修改指标重算。


In [8]:
def run_one_holdout(heldout_id, run_dir):
    started = time.perf_counter()
    fold_dir = run_dir / f"holdout_{heldout_id}"
    fold_dir.mkdir(parents=True, exist_ok=False)
    checkpoint_dir = fold_dir / "checkpoints"
    checkpoint_dir.mkdir()
    train_ids = training_ids(heldout_id)
    print(f"\n=== 留出 {heldout_id}；训练 {train_ids}；目录 {fold_dir} ===")
    pl.seed_everything(SEED, workers=True)

    # 此时仅加载训练切片，留出文件不参与任何预处理。
    train_by_id = {i: read_slice(i) for i in train_ids}
    train_adatas = [train_by_id[i] for i in train_ids]
    genes = train_adatas[0].var_names
    if not all(genes.equals(a.var_names) for a in train_adatas):
        raise ValueError("训练切片基因列表或顺序不一致")
    train_counts = pd.concat([a.obs[LABEL_KEY] for a in train_adatas]).value_counts()
    train_counts = train_counts[train_counts > 0]
    train_counts.rename_axis("cell_class").rename("train_n").to_csv(fold_dir / "train_cell_counts.csv")
    fold_parameters = dict(
        heldout_id=heldout_id, train_ids=train_ids, reference_pair=[heldout_id - 1, heldout_id + 1],
        input_files={str(i): str(SLICE_FILES[i]) for i in train_ids},
        heldout_file=str(SLICE_FILES[heldout_id]), seed=SEED, epochs=EPOCHS,
        slice_gap_um=SLICE_GAP_UM, thickness_um=THICKNESS_UM, slab_half_width_um=SLAB_HALF_WIDTH_UM,
        ode_steps=ODE_STEPS, chunk_size=CHUNK_SIZE,
        rare_train_fraction=RARE_TRAIN_FRACTION, coverage_radii_um=list(COVERAGE_RADII_UM),
        data_kwargs=DATA_KWARGS, model_kwargs=MODEL_KWARGS,
        inference_checkpoint="checkpoints/last.ckpt", inference_weights="EMA",
        torch_version=str(torch.__version__), cuda_version=torch.version.cuda,
        lightning_version=str(pl.__version__), gpu=torch.cuda.get_device_name(0),
    )
    write_json(fold_dir / "run_parameters.json", fold_parameters)

    model = DeepSpatial()  # 每轮全新模型；不复用五张切片训练过的权重
    model.setup_data(adata_list=train_adatas, **DATA_KWARGS)
    model.build_model(**MODEL_KWARGS)
    model._save_config(str(checkpoint_dir))  # 与 core.py 中 fit(save_ckpt=True) 保持一致
    checkpoint = ModelCheckpoint(
        dirpath=str(checkpoint_dir), monitor="loss", mode="min",
        filename="deepspatial-{epoch:02d}-{loss:.4f}", save_top_k=1, save_last=True,
    )
    logger = CSVLogger(save_dir=str(fold_dir), name="logs", version=0)
    trainer = pl.Trainer(
        max_epochs=EPOCHS, accelerator="gpu", devices=1,
        callbacks=[checkpoint], logger=logger, enable_checkpointing=True,
        default_root_dir=str(fold_dir), log_every_n_steps=10,
    )
    trainer.fit(model.module, model.train_loader)
    if not (checkpoint_dir / "last.ckpt").is_file():
        raise RuntimeError("未生成 last.ckpt，请检查训练是否正常结束")

    # 仅重建跨过留出位置的区间；两端均为训练切片。
    gap = model.reconstruct_between_slices(
        adata0=train_by_id[heldout_id - 1], adata1=train_by_id[heldout_id + 1],
        thickness=THICKNESS_UM, steps=ODE_STEPS, chunk_size=CHUNK_SIZE, device="cuda:0",
    )
    # 非建模 obs 列来自源细胞，显式改名避免误当作预测注释。
    inherited = {c: f"source_{c}" for c in gap.obs.columns if c not in (LABEL_KEY, "z_coord")}
    gap.obs = gap.obs.rename(columns=inherited)
    add_spatial_3d(gap)
    gap.uns["holdout_protocol"] = dict(heldout_id=heldout_id, train_ids=train_ids,
                                      slab_half_width_um=SLAB_HALF_WIDTH_UM)
    gap.write_h5ad(fold_dir / "predicted_gap.h5ad", compression="gzip")
    center_z = heldout_id * SLICE_GAP_UM
    mask = np.abs(gap.obs["z_coord"].to_numpy() - center_z) <= SLAB_HALF_WIDTH_UM
    prediction = gap[mask].copy()
    if prediction.n_obs == 0:
        raise RuntimeError("固定宽度薄层中没有生成细胞，请检查区间生成。")
    prediction.write_h5ad(fold_dir / "predicted_holdout.h5ad", compression="gzip")

    # 从这里开始才读取留出真值，后续不再训练或调整生成参数。
    truth = add_spatial_3d(read_slice(heldout_id))
    truth.write_h5ad(fold_dir / "heldout_truth.h5ad", compression="gzip")
    table, summary = evaluate_holdout(truth, prediction, train_counts)
    table.insert(0, "heldout_id", heldout_id)
    table.to_csv(fold_dir / "cell_type_metrics.csv", index=False)
    rare_table = table[table.is_rare_train | table.unseen_in_training].copy()
    rare_table.to_csv(fold_dir / "rare_cell_metrics.csv", index=False)
    summary = dict(heldout_id=heldout_id, train_ids=",".join(map(str, train_ids)),
                   seed=SEED, gap_generated_n=int(gap.n_obs), **summary,
                   elapsed_seconds=float(time.perf_counter() - started), output_dir=str(fold_dir))
    write_json(fold_dir / "metrics.json", summary)
    print(f"完成留出 {heldout_id}：真值 {truth.n_obs}，预测薄层 {prediction.n_obs}，"
          f"稀有类缺失 {summary['rare_missing_classes']}；用时 {summary['elapsed_seconds'] / 60:.1f} 分钟")
    return summary, table


## 开始连续验证

下面的 cell 会实际启动训练，默认顺序运行三轮，每轮 100 epochs。
每次执行此 cell 都创建一个带时间戳的新目录；中途失败时，已完成轮次的文件和汇总仍保留。
三轮是三个切片划分，不是三个独立种子的重复实验；汇总的标准差描述切片间差异。


In [9]:
RUN_DIR = OUTPUT_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
write_json(RUN_DIR / "run_parameters.json", dict(
    holdout_slice_ids=HOLDOUT_SLICE_IDS, seed=SEED, epochs=EPOCHS,
    protocol="leave_one_middle_slice_out", output_dir=str(RUN_DIR),
))
print("本次全部结果目录：", RUN_DIR)

fold_summaries, fold_tables = [], []
for heldout_id in HOLDOUT_SLICE_IDS:
    summary, table = run_one_holdout(heldout_id, RUN_DIR)
    fold_summaries.append(summary)
    fold_tables.append(table)
    summary_df = pd.DataFrame(fold_summaries)
    summary_df.to_csv(RUN_DIR / "fold_summary.csv", index=False)
    all_classes = pd.concat(fold_tables, ignore_index=True)
    all_classes.to_csv(RUN_DIR / "all_cell_type_metrics.csv", index=False)
    all_classes[all_classes.is_rare_train | all_classes.unseen_in_training].to_csv(
        RUN_DIR / "all_rare_cell_metrics.csv", index=False,
    )
    # 函数返回后释放本轮模型、优化器和 GPU 缓存，下一轮从头初始化。
    gc.collect()
    torch.cuda.empty_cache()

display(summary_df)


Seed set to 42


本次全部结果目录： /private_sdb/huangyingjie/Code/DeepSpatial/output/merfish_holdout/20261018_021947_653667

=== 留出 1；训练 [0, 2, 3, 4]；目录 /private_sdb/huangyingjie/Code/DeepSpatial/output/merfish_holdout/20261018_021947_653667/holdout_1 ===


DeepSpatial: Building Trajectories: 100%|██████████| 3/3 [00:23<00:00,  7.88s/it]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/private_sdb/huangyingjie/miniconda3/envs/deepspatial/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /private_sdb/huangyingjie/Code/DeepSpatial/output/merfish_holdout/20261018_021947_653667/holdout_1/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type | Params | Mode 
-------------------------------------------
0 | model     | GiT 

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


...

DeepSpatial: Building Trajectories: 100%|██████████| 3/3 [00:17<00:00,  5.73s/it]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/private_sdb/huangyingjie/miniconda3/envs/deepspatial/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /private_sdb/huangyingjie/Code/DeepSpatial/output/merfish_holdout/20261018_021947_653667/holdout_3/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type | Params | Mode 
-------------------------------------------
0 | model     | GiT  | 7.7 M  | train
1 | ema_model | GiT  | 7.7 M  | eval 
-------------------------------------------
7.7 M     Trainable params
7.7 M     Non-trainable params
15.4 M    Total params
61.696    Total estimated model params size (MB)
152       Modules in train mode
152       Modules in eval mode
/private_sdb/huangyingjie/miniconda3/envs/deepspatial/lib/python3.10/si

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


完成留出 3：真值 5803，预测薄层 5763，稀有类缺失 0；用时 15.9 分钟


,heldout_id,train_ids,seed,gap_generated_n,truth_n,predicted_n,predicted_to_truth_count_ratio,truth_classes,predicted_classes,missing_classes,...,global_mean_log_expression_pcc,negative_prediction_fraction,rare_macro_coverage_recall_25um,rare_macro_coverage_precision_25um,rare_macro_coverage_recall_50um,rare_macro_coverage_precision_50um,rare_macro_coverage_recall_100um,rare_macro_coverage_precision_100um,elapsed_seconds,output_dir
0,1,"0,2,3,4",42,57070,5557,5686,1.023214,15,15,0,...,0.977707,0.000008,0.029073,0.023039,0.082731,0.086995,0.216379,0.218206,1037.374922,/private_sdb/huangyingjie/Code/DeepSpatial/out...
1,2,"0,1,3,4",42,56800,5926,5630,0.950051,15,15,0,...,0.996619,0.000000,0.003876,0.004630,0.055106,0.053880,0.190057,0.239417,1083.506577,/private_sdb/huangyingjie/Code/DeepSpatial/out...
2,3,"0,1,2,4",42,57345,5803,5763,0.993107,15,15,0,...,0.993607,0.000000,0.020666,0.025951,0.061398,0.049698,0.210060,0.119712,956.861367,/private_sdb/huangyingjie/Code/DeepSpatial/out...


## 查看结果

`fold_summary.csv` 是各留出切片的总体指标；`all_rare_cell_metrics.csv` 可定位具体稀有类。

每个 `holdout_编号` 子目录保存：

| 文件 | 内容 |
|---|---|
| `predicted_gap.h5ad` | 留出切片两侧之间的整个生成区间 |
| `predicted_holdout.h5ad` | 留出 Z 位置 ±5 µm 的预测薄层，用于指标 |
| `heldout_truth.h5ad` | 真实留出切片，用于评估 |
| `cell_type_metrics.csv` / `rare_cell_metrics.csv` | 逐类数量、比例、空间覆盖率、平均表达 PCC |
| `metrics.json` / `run_parameters.json` | 整体指标 / 本轮完整参数与划分 |
| `train_cell_counts.csv` | 四张训练切片的类别数量，稀有类定义的依据 |
| `checkpoints/last.ckpt`、最佳训练损失 ckpt、`config.json` | 模型及训练元数据 |
| `logs/version_0/metrics.csv` | 训练损失日志，包含 step / epoch；不是留出验证损失 |

三个 h5ad 均有 `obsm['spatial']`（XY）、`obs['z_coord']` 和 `obsm['spatial_3d']`（XYZ）。
`cell_class` 是被建模的类别；预测中的 `source_*` 仅说明继承的源细胞属性。
先看 `missing_in_prediction` 判断是否整类没有生成，再看 `coverage_recall_50um` / `coverage_recall_100um` 判断局部覆盖情况。
训练未见类别单独解释；在没有逐细胞配对真值的情况下，不把最近邻覆盖率称为真实细胞存活率。


In [10]:
key_columns = [
    "heldout_id", "truth_n", "predicted_n", "rare_missing_classes", "unseen_training_classes",
    "rare_macro_coverage_recall_50um", "rare_macro_coverage_recall_100um",
    "global_mean_log_expression_pcc", "symmetric_mean_nn_um",
]
display(summary_df[key_columns])
rare_results = all_classes[(all_classes.is_rare_train | all_classes.unseen_in_training) & (all_classes.truth_n > 0)]
display(rare_results[[
    "heldout_id", "cell_class", "train_n", "truth_n", "predicted_n", "missing_in_prediction",
    "unseen_in_training", "coverage_recall_50um", "coverage_recall_100um", "mean_log_expression_pcc",
]])
if len(summary_df) > 1:
    numeric_columns = [c for c in key_columns if c != "heldout_id"]
    aggregate = summary_df[numeric_columns].agg(["mean", "std"]).T
    aggregate.to_csv(RUN_DIR / "fold_mean_std.csv")
    display(aggregate)
print("结果已保存：", RUN_DIR)


,heldout_id,truth_n,predicted_n,rare_missing_classes,unseen_training_classes,rare_macro_coverage_recall_50um,rare_macro_coverage_recall_100um,global_mean_log_expression_pcc,symmetric_mean_nn_um
0,1,5557,5686,0,0,0.082731,0.216379,0.977707,14.763772
1,2,5926,5630,0,0,0.055106,0.190057,0.996619,17.194286
2,3,5803,5763,0,0,0.061398,0.210060,0.993607,14.563923


,heldout_id,cell_class,train_n,truth_n,predicted_n,missing_in_prediction,unseen_in_training,coverage_recall_50um,coverage_recall_100um,mean_log_expression_pcc
2,1,Endothelial 2,103,41,31,False,False,0.146341,0.414634,0.935416
9,1,OD Immature 2,28,5,5,False,False,0.200000,0.200000,0.686181
10,1,OD Mature 1,209,29,69,False,False,0.068966,0.241379,0.964537
12,1,OD Mature 3,11,3,2,False,False,0.000000,0.000000,0.811935
13,1,OD Mature 4,36,11,10,False,False,0.000000,0.090909,0.897971
14,1,Pericytes,167,37,32,False,False,0.081081,0.351351,0.969919
17,2,Endothelial 2,127,17,25,False,False,0.176471,0.352941,0.918118
24,2,OD Immature 2,28,5,4,False,False,0.000000,0.200000,0.710202
25,2,OD Mature 1,195,43,36,False,False,0.023256,0.325581,0.972183
27,2,OD Mature 3,12,2,4,False,False,0.000000,0.000000,0.828436


,mean,std
truth_n,5762.000000,187.885603
predicted_n,5693.000000,66.775744
rare_missing_classes,0.000000,0.000000
unseen_training_classes,0.000000,0.000000
rare_macro_coverage_recall_50um,0.066412,0.014479
rare_macro_coverage_recall_100um,0.205499,0.013741
global_mean_log_expression_pcc,0.989311,0.010162
symmetric_mean_nn_um,15.507327,1.464363


结果已保存： /private_sdb/huangyingjie/Code/DeepSpatial/output/merfish_holdout/20261018_021947_653667
